In [1]:
# ================================================================
# NewsGuard — Day 3
# Cell 1: Load Preprocessed Datasets
# ================================================================

from google.colab import drive
drive.mount('/content/drive')

import os
import pandas as pd

print("=" * 64)
print("NewsGuard — Day 3")
print("TF-IDF Feature Engineering")
print("=" * 64)

processed_dir = "/content/drive/MyDrive/NewsGuard/data/processed"

train_path = os.path.join(
    processed_dir,
    "train_preprocessed.csv"
)

val_path = os.path.join(
    processed_dir,
    "validation_preprocessed.csv"
)

test_path = os.path.join(
    processed_dir,
    "test_preprocessed.csv"
)

# Verify files
for path in [train_path, val_path, test_path]:
    print(f"{'FOUND' if os.path.exists(path) else 'MISSING'} : {path}")
    assert os.path.exists(path)

# Load datasets
train_df = pd.read_csv(train_path)
val_df = pd.read_csv(val_path)
test_df = pd.read_csv(test_path)

print("\n" + "=" * 64)
print("Datasets Loaded")
print("=" * 64)

print(f"Train      : {train_df.shape}")
print(f"Validation : {val_df.shape}")
print(f"Test       : {test_df.shape}")

print("\nColumns:")
print(train_df.columns.tolist())

print("\nLabel distribution:")
print("Train:")
print(train_df["label"].value_counts().sort_index())

print("\nValidation:")
print(val_df["label"].value_counts().sort_index())

print("\nTest:")
print(test_df["label"].value_counts().sort_index())

Mounted at /content/drive
NewsGuard — Day 3
TF-IDF Feature Engineering
FOUND : /content/drive/MyDrive/NewsGuard/data/processed/train_preprocessed.csv
FOUND : /content/drive/MyDrive/NewsGuard/data/processed/validation_preprocessed.csv
FOUND : /content/drive/MyDrive/NewsGuard/data/processed/test_preprocessed.csv

Datasets Loaded
Train      : (31282, 7)
Validation : (3910, 7)
Test       : (3911, 7)

Columns:
['title', 'text', 'label', 'content', 'word_count', 'char_count', 'clean_content']

Label distribution:
Train:
label
0    14325
1    16957
Name: count, dtype: int64

Validation:
label
0    1791
1    2119
Name: count, dtype: int64

Test:
label
0    1791
1    2120
Name: count, dtype: int64


In [2]:
# ================================================================
# NewsGuard — Day 3
# Cell 2: Separate Text Features and Labels
# ================================================================

# Text features
X_train_text = train_df["clean_content"].astype(str)
X_val_text = val_df["clean_content"].astype(str)
X_test_text = test_df["clean_content"].astype(str)

# Target labels
y_train = train_df["label"].astype(int)
y_val = val_df["label"].astype(int)
y_test = test_df["label"].astype(int)

print("=" * 64)
print("TEXT FEATURES & LABELS PREPARED")
print("=" * 64)

print("\nText data:")
print(f"X_train_text : {X_train_text.shape}")
print(f"X_val_text   : {X_val_text.shape}")
print(f"X_test_text  : {X_test_text.shape}")

print("\nLabels:")
print(f"y_train : {y_train.shape}")
print(f"y_val   : {y_val.shape}")
print(f"y_test  : {y_test.shape}")

print("\nLabel values:")
print(f"Train      : {sorted(y_train.unique())}")
print(f"Validation : {sorted(y_val.unique())}")
print(f"Test       : {sorted(y_test.unique())}")

# Integrity checks
assert len(X_train_text) == len(y_train)
assert len(X_val_text) == len(y_val)
assert len(X_test_text) == len(y_test)

assert set(y_train.unique()) == {0, 1}
assert set(y_val.unique()) == {0, 1}
assert set(y_test.unique()) == {0, 1}

print("\n" + "=" * 64)
print("TEXT & LABEL SEPARATION PASSED")
print("=" * 64)

TEXT FEATURES & LABELS PREPARED

Text data:
X_train_text : (31282,)
X_val_text   : (3910,)
X_test_text  : (3911,)

Labels:
y_train : (31282,)
y_val   : (3910,)
y_test  : (3911,)

Label values:
Train      : [np.int64(0), np.int64(1)]
Validation : [np.int64(0), np.int64(1)]
Test       : [np.int64(0), np.int64(1)]

TEXT & LABEL SEPARATION PASSED


In [3]:
# ================================================================
# NewsGuard — Day 3
# Cell 3: TF-IDF Vectorizer Configuration
# ================================================================

from sklearn.feature_extraction.text import TfidfVectorizer

print("=" * 64)
print("CONFIGURING TF-IDF VECTORIZER")
print("=" * 64)

tfidf_vectorizer = TfidfVectorizer(
    max_features=5000,
    ngram_range=(1, 2),
    min_df=2,
    max_df=0.95,
    sublinear_tf=True,
    dtype="float32"
)

print("\nTF-IDF Configuration:")
print("-" * 64)
print(f"max_features : {tfidf_vectorizer.max_features}")
print(f"ngram_range  : {tfidf_vectorizer.ngram_range}")
print(f"min_df       : {tfidf_vectorizer.min_df}")
print(f"max_df       : {tfidf_vectorizer.max_df}")
print(f"sublinear_tf : {tfidf_vectorizer.sublinear_tf}")
print(f"dtype        : {tfidf_vectorizer.dtype}")

print("\n" + "=" * 64)
print("TF-IDF VECTORIZER CONFIGURED")
print("=" * 64)

CONFIGURING TF-IDF VECTORIZER

TF-IDF Configuration:
----------------------------------------------------------------
max_features : 5000
ngram_range  : (1, 2)
min_df       : 2
max_df       : 0.95
sublinear_tf : True
dtype        : float32

TF-IDF VECTORIZER CONFIGURED


In [4]:
# ================================================================
# NewsGuard — Day 3
# Cell 4: Fit TF-IDF on Training Data Only
# ================================================================

print("=" * 64)
print("FITTING TF-IDF ON TRAINING DATA")
print("=" * 64)

# IMPORTANT:
# TF-IDF vocabulary + IDF weights are learned ONLY from training data.
X_train_tfidf = tfidf_vectorizer.fit_transform(X_train_text)

print("\nTF-IDF Training Matrix:")
print(f"Shape          : {X_train_tfidf.shape}")
print(f"Rows           : {X_train_tfidf.shape[0]:,}")
print(f"Features       : {X_train_tfidf.shape[1]:,}")
print(f"Matrix type    : {type(X_train_tfidf).__name__}")
print(f"Data type      : {X_train_tfidf.dtype}")

print("\n" + "-" * 64)
print("Vocabulary Information")
print("-" * 64)

vocab_size = len(tfidf_vectorizer.vocabulary_)

print(f"Vocabulary size : {vocab_size:,}")

# Leakage check
assert X_train_tfidf.shape[0] == len(y_train)
assert X_train_tfidf.shape[1] <= 5000
assert vocab_size == X_train_tfidf.shape[1]

print("\n" + "=" * 64)
print("TRAINING TF-IDF FIT COMPLETED")
print("TF-IDF learned from TRAINING DATA ONLY")
print("=" * 64)

FITTING TF-IDF ON TRAINING DATA


/usr/local/lib/python3.13/dist-packages/sklearn/feature_extraction/text.py:2043: UserWarning: Only (<class 'numpy.float64'>, <class 'numpy.float32'>, <class 'numpy.float16'>) 'dtype' should be used. float32 'dtype' will be converted to np.float64.
  warnings.warn(



TF-IDF Training Matrix:
Shape          : (31282, 5000)
Rows           : 31,282
Features       : 5,000
Matrix type    : csr_matrix
Data type      : float32

----------------------------------------------------------------
Vocabulary Information
----------------------------------------------------------------
Vocabulary size : 5,000

TRAINING TF-IDF FIT COMPLETED
TF-IDF learned from TRAINING DATA ONLY


In [5]:
# ================================================================
# NewsGuard — Day 3
# Cell 5: Transform Validation & Test Data
# ================================================================

print("=" * 64)
print("TRANSFORMING VALIDATION & TEST DATA")
print("=" * 64)

# IMPORTANT:
# Do NOT fit the vectorizer again.
# Use the vocabulary and IDF learned from training data.

X_val_tfidf = tfidf_vectorizer.transform(X_val_text)
X_test_tfidf = tfidf_vectorizer.transform(X_test_text)

print("\nTF-IDF Matrices:")
print("-" * 64)

print(f"Train      : {X_train_tfidf.shape}")
print(f"Validation : {X_val_tfidf.shape}")
print(f"Test       : {X_test_tfidf.shape}")

print("\nMatrix types:")
print(f"Train      : {type(X_train_tfidf).__name__}")
print(f"Validation : {type(X_val_tfidf).__name__}")
print(f"Test       : {type(X_test_tfidf).__name__}")

print("\nFeature counts:")
print(f"Train      : {X_train_tfidf.shape[1]:,}")
print(f"Validation : {X_val_tfidf.shape[1]:,}")
print(f"Test       : {X_test_tfidf.shape[1]:,}")

# Integrity checks
assert X_train_tfidf.shape == (31282, 5000)
assert X_val_tfidf.shape == (3910, 5000)
assert X_test_tfidf.shape == (3911, 5000)

assert X_train_tfidf.shape[1] == X_val_tfidf.shape[1]
assert X_train_tfidf.shape[1] == X_test_tfidf.shape[1]

assert X_train_tfidf.shape[0] == len(y_train)
assert X_val_tfidf.shape[0] == len(y_val)
assert X_test_tfidf.shape[0] == len(y_test)

print("\n" + "=" * 64)
print("VALIDATION & TEST TF-IDF TRANSFORMATION PASSED")
print("Same TRAIN-learned vocabulary used for all splits.")
print("=" * 64)

TRANSFORMING VALIDATION & TEST DATA

TF-IDF Matrices:
----------------------------------------------------------------
Train      : (31282, 5000)
Validation : (3910, 5000)
Test       : (3911, 5000)

Matrix types:
Train      : csr_matrix
Validation : csr_matrix
Test       : csr_matrix

Feature counts:
Train      : 5,000
Validation : 5,000
Test       : 5,000

VALIDATION & TEST TF-IDF TRANSFORMATION PASSED
Same TRAIN-learned vocabulary used for all splits.


In [6]:
# ================================================================
# NewsGuard — Day 3
# Cell 6: Inspect TF-IDF Vocabulary
# ================================================================

feature_names = tfidf_vectorizer.get_feature_names_out()

print("=" * 64)
print("TF-IDF VOCABULARY INSPECTION")
print("=" * 64)

print(f"\nTotal features : {len(feature_names):,}")

print("\nFirst 50 features:")
print("-" * 64)

for i, feature in enumerate(feature_names[:50], start=1):
    print(f"{i:02d}. {feature}")

# Check unigram and bigram presence
unigrams = sum(" " not in feature for feature in feature_names)
bigrams = sum(" " in feature for feature in feature_names)

print("\n" + "-" * 64)
print("N-gram composition")
print("-" * 64)
print(f"Unigrams : {unigrams:,}")
print(f"Bigrams  : {bigrams:,}")

assert len(feature_names) == 5000
assert unigrams > 0
assert bigrams > 0

print("\n" + "=" * 64)
print("TF-IDF VOCABULARY CHECK PASSED")
print("Unigrams + bigrams successfully generated.")
print("=" * 64)

TF-IDF VOCABULARY INSPECTION

Total features : 5,000

First 50 features:
----------------------------------------------------------------
01. 000
02. 000 people
03. 10
04. 100
05. 100 000
06. 11
07. 12
08. 13
09. 14
10. 15
11. 16
12. 17
13. 18
14. 19
15. 20
16. 200
17. 2001
18. 2005
19. 2006
20. 2007
21. 2008
22. 2009
23. 2010
24. 2011
25. 2012
26. 2013
27. 2014
28. 2015
29. 2016
30. 2016 election
31. 2016 presidential
32. 2017
33. 2017 realdonaldtrump
34. 2018
35. 2019
36. 2020
37. 21
38. 21st
39. 21st century
40. 21wire
41. 22
42. 23
43. 24
44. 25
45. 26
46. 27
47. 28
48. 29
49. 30
50. 300

----------------------------------------------------------------
N-gram composition
----------------------------------------------------------------
Unigrams : 2,957
Bigrams  : 2,043

TF-IDF VOCABULARY CHECK PASSED
Unigrams + bigrams successfully generated.


In [7]:
# ================================================================
# NewsGuard — Day 3
# Cell 7: TF-IDF Matrix Statistics
# ================================================================

print("=" * 64)
print("TF-IDF MATRIX STATISTICS")
print("=" * 64)

matrices = [
    (X_train_tfidf, "TRAIN"),
    (X_val_tfidf, "VALIDATION"),
    (X_test_tfidf, "TEST")
]

for matrix, name in matrices:
    total_elements = matrix.shape[0] * matrix.shape[1]
    non_zero = matrix.nnz
    sparsity = 1 - (non_zero / total_elements)
    density = non_zero / total_elements

    print(f"\n{name}")
    print("-" * 64)
    print(f"Shape             : {matrix.shape}")
    print(f"Non-zero values   : {non_zero:,}")
    print(f"Sparsity          : {sparsity:.4%}")
    print(f"Density            : {density:.4%}")
    print(f"Memory format     : {matrix.format}")
    print(f"Data type         : {matrix.dtype}")

    assert matrix.nnz > 0
    assert 0 < density < 1
    assert 0 < sparsity < 1

print("\n" + "=" * 64)
print("TF-IDF MATRIX STATISTICS CHECK PASSED")
print("=" * 64)

TF-IDF MATRIX STATISTICS

TRAIN
----------------------------------------------------------------
Shape             : (31282, 5000)
Non-zero values   : 7,251,777
Sparsity          : 95.3636%
Density            : 4.6364%
Memory format     : csr
Data type         : float32

VALIDATION
----------------------------------------------------------------
Shape             : (3910, 5000)
Non-zero values   : 904,827
Sparsity          : 95.3717%
Density            : 4.6283%
Memory format     : csr
Data type         : float32

TEST
----------------------------------------------------------------
Shape             : (3911, 5000)
Non-zero values   : 899,935
Sparsity          : 95.3979%
Density            : 4.6021%
Memory format     : csr
Data type         : float32

TF-IDF MATRIX STATISTICS CHECK PASSED


In [8]:
# ================================================================
# NewsGuard — Day 3
# Cell 8: Inspect Top TF-IDF Terms
# ================================================================

import numpy as np

print("=" * 64)
print("TOP TF-IDF TERMS")
print("=" * 64)

feature_names = tfidf_vectorizer.get_feature_names_out()

# Average TF-IDF score of each feature across training documents
mean_tfidf = np.asarray(
    X_train_tfidf.mean(axis=0)
).ravel()

top_indices = np.argsort(mean_tfidf)[::-1][:30]

print("\nTop 30 features by average TF-IDF score:")
print("-" * 64)

for rank, idx in enumerate(top_indices, start=1):
    print(
        f"{rank:02d}. "
        f"{feature_names[idx]:<35} "
        f"{mean_tfidf[idx]:.6f}"
    )

assert len(top_indices) == 30
assert np.all(mean_tfidf >= 0)

print("\n" + "=" * 64)
print("TOP TF-IDF TERM INSPECTION PASSED")
print("=" * 64)

TOP TF-IDF TERMS

Top 30 features by average TF-IDF score:
----------------------------------------------------------------
01. of                                  0.045180
02. and                                 0.043513
03. in                                  0.042250
04. on                                  0.035324
05. that                                0.035287
06. trump                               0.033145
07. for                                 0.032267
08. is                                  0.030967
09. said                                0.030668
10. he                                  0.028656
11. it                                  0.028311
12. with                                0.027375
13. was                                 0.026301
14. as                                  0.025426
15. his                                 0.025187
16. of the                              0.024819
17. by                                  0.024489
18. has                                 0.0

In [9]:
# ================================================================
# NewsGuard — Day 3
# Cell 9: Save TF-IDF Matrices & Vectorizer
# ================================================================

from scipy.sparse import save_npz
import joblib
import os

tfidf_dir = "/content/drive/MyDrive/NewsGuard/features/tfidf"
os.makedirs(tfidf_dir, exist_ok=True)

train_tfidf_path = os.path.join(
    tfidf_dir,
    "X_train_tfidf.npz"
)

val_tfidf_path = os.path.join(
    tfidf_dir,
    "X_validation_tfidf.npz"
)

test_tfidf_path = os.path.join(
    tfidf_dir,
    "X_test_tfidf.npz"
)

vectorizer_path = os.path.join(
    tfidf_dir,
    "tfidf_vectorizer.joblib"
)

# Save sparse matrices
save_npz(train_tfidf_path, X_train_tfidf)
save_npz(val_tfidf_path, X_val_tfidf)
save_npz(test_tfidf_path, X_test_tfidf)

# Save fitted vectorizer
joblib.dump(tfidf_vectorizer, vectorizer_path)

print("=" * 64)
print("TF-IDF ARTIFACTS SAVED")
print("=" * 64)

for path in [
    train_tfidf_path,
    val_tfidf_path,
    test_tfidf_path,
    vectorizer_path
]:
    print(f"{'FOUND' if os.path.exists(path) else 'MISSING'} : {path}")
    assert os.path.exists(path)

print("\n" + "=" * 64)
print("TF-IDF ARTIFACT SAVE PASSED")
print("=" * 64)

TF-IDF ARTIFACTS SAVED
FOUND : /content/drive/MyDrive/NewsGuard/features/tfidf/X_train_tfidf.npz
FOUND : /content/drive/MyDrive/NewsGuard/features/tfidf/X_validation_tfidf.npz
FOUND : /content/drive/MyDrive/NewsGuard/features/tfidf/X_test_tfidf.npz
FOUND : /content/drive/MyDrive/NewsGuard/features/tfidf/tfidf_vectorizer.joblib

TF-IDF ARTIFACT SAVE PASSED


In [10]:
# ================================================================
# NewsGuard — Day 3
# Cell 10: Reload & Verify TF-IDF Artifacts
# ================================================================

from scipy.sparse import load_npz
import joblib
import numpy as np

# Reload matrices
X_train_tfidf_loaded = load_npz(train_tfidf_path)
X_val_tfidf_loaded = load_npz(val_tfidf_path)
X_test_tfidf_loaded = load_npz(test_tfidf_path)

# Reload vectorizer
tfidf_vectorizer_loaded = joblib.load(vectorizer_path)

print("=" * 64)
print("TF-IDF ARTIFACT RELOAD VERIFICATION")
print("=" * 64)

print("\nReloaded shapes:")
print(f"Train      : {X_train_tfidf_loaded.shape}")
print(f"Validation : {X_val_tfidf_loaded.shape}")
print(f"Test       : {X_test_tfidf_loaded.shape}")

# Exact structural checks
assert X_train_tfidf_loaded.shape == X_train_tfidf.shape
assert X_val_tfidf_loaded.shape == X_val_tfidf.shape
assert X_test_tfidf_loaded.shape == X_test_tfidf.shape

# Numerical equality checks
assert np.array_equal(
    X_train_tfidf_loaded.toarray(),
    X_train_tfidf.toarray()
)

assert np.array_equal(
    X_val_tfidf_loaded.toarray(),
    X_val_tfidf.toarray()
)

assert np.array_equal(
    X_test_tfidf_loaded.toarray(),
    X_test_tfidf.toarray()
)

# Vectorizer verification
assert len(tfidf_vectorizer_loaded.get_feature_names_out()) == 5000

print("\nMatrix equality:")
print("Train      : PASSED")
print("Validation : PASSED")
print("Test       : PASSED")

print("\nVectorizer:")
print(f"Features   : {len(tfidf_vectorizer_loaded.get_feature_names_out()):,}")

print("\n" + "=" * 64)
print("TF-IDF RELOAD VERIFICATION PASSED")
print("Saved artifacts are identical to the original artifacts.")
print("=" * 64)

TF-IDF ARTIFACT RELOAD VERIFICATION

Reloaded shapes:
Train      : (31282, 5000)
Validation : (3910, 5000)
Test       : (3911, 5000)

Matrix equality:
Train      : PASSED
Validation : PASSED
Test       : PASSED

Vectorizer:
Features   : 5,000

TF-IDF RELOAD VERIFICATION PASSED
Saved artifacts are identical to the original artifacts.


In [11]:
# ================================================================
# NewsGuard — Day 3
# Cell 11: Final TF-IDF Integrity Check
# ================================================================

import os

print("=" * 64)
print("DAY 3 — FINAL TF-IDF INTEGRITY CHECK")
print("=" * 64)

# Dataset sizes
assert X_train_tfidf.shape[0] == len(train_df)
assert X_val_tfidf.shape[0] == len(val_df)
assert X_test_tfidf.shape[0] == len(test_df)

# Feature consistency
assert X_train_tfidf.shape[1] == 5000
assert X_val_tfidf.shape[1] == 5000
assert X_test_tfidf.shape[1] == 5000

# Sparse matrices
assert hasattr(X_train_tfidf, "tocsr")
assert hasattr(X_val_tfidf, "tocsr")
assert hasattr(X_test_tfidf, "tocsr")

# Vectorizer
assert len(tfidf_vectorizer.get_feature_names_out()) == 5000

# Saved artifacts
required_artifacts = [
    train_tfidf_path,
    val_tfidf_path,
    test_tfidf_path,
    vectorizer_path
]

for artifact in required_artifacts:
    assert os.path.exists(artifact)

print("\nDataset rows:")
print(f"Train      : {len(train_df):,}")
print(f"Validation : {len(val_df):,}")
print(f"Test       : {len(test_df):,}")

print("\nTF-IDF matrices:")
print(f"Train      : {X_train_tfidf.shape}")
print(f"Validation : {X_val_tfidf.shape}")
print(f"Test       : {X_test_tfidf.shape}")

print("\nConfiguration:")
print("Max features : 5,000")
print("N-grams      : (1, 2)")
print("min_df       : 2")
print("max_df       : 0.95")
print("sublinear_tf : True")

print("\nArtifacts:")
for artifact in required_artifacts:
    print(f"FOUND : {artifact}")

print("\n" + "=" * 64)
print("DAY 3 TF-IDF INTEGRITY CHECK PASSED")
print("TF-IDF FEATURE ENGINEERING COMPLETE.")
print("Ready for the next feature-engineering stage.")
print("=" * 64)

DAY 3 — FINAL TF-IDF INTEGRITY CHECK

Dataset rows:
Train      : 31,282
Validation : 3,910
Test       : 3,911

TF-IDF matrices:
Train      : (31282, 5000)
Validation : (3910, 5000)
Test       : (3911, 5000)

Configuration:
Max features : 5,000
N-grams      : (1, 2)
min_df       : 2
max_df       : 0.95
sublinear_tf : True

Artifacts:
FOUND : /content/drive/MyDrive/NewsGuard/features/tfidf/X_train_tfidf.npz
FOUND : /content/drive/MyDrive/NewsGuard/features/tfidf/X_validation_tfidf.npz
FOUND : /content/drive/MyDrive/NewsGuard/features/tfidf/X_test_tfidf.npz
FOUND : /content/drive/MyDrive/NewsGuard/features/tfidf/tfidf_vectorizer.joblib

DAY 3 TF-IDF INTEGRITY CHECK PASSED
TF-IDF FEATURE ENGINEERING COMPLETE.
Ready for the next feature-engineering stage.
